---
**Non-Parametric & Proportion Tests in Python**
Data Analysis Course · Week 8
---

This notebook is the Python equivalent of the R Markdown `_07_hypothesis_testing_prop.Rmd`.
Topics: **non-parametric tests** (Wilcoxon/Mann-Whitney), **proportion tests** (Fisher's exact test,
chi-square), and the **power of a test**.

Work through it cell by cell — run each code cell with **Shift+Enter**.

**Required packages:** `numpy`, `pandas`, `matplotlib`, `scipy`, `pyreadr`
```
pip install numpy pandas matplotlib scipy pyreadr
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import pyreadr, urllib.request

We work with tumor expression data of AML/ALL leukemia patients:

In [ ]:
all_aml = pd.read_csv(
    "https://www.dropbox.com/scl/fi/q4rbpl3un6oqbbwoxe1qw/all.aml.cleaned.csv?rlkey=igwbna22d6hpdmw3os79zoy7z&dl=1",
    sep="\t"
)
all_aml = all_aml.set_index(all_aml.columns[0])   # R: data.matrix(all.aml), genes as rows

urllib.request.urlretrieve(
    "https://www.dropbox.com/scl/fi/0lqjh3jcskki2bb8hxwj6/all.aml.anno.rds?rlkey=cijsx39kr9gx9bjl6wzpmkgnz&dl=1",
    "all_aml_anno.rds"
)
all_aml_anno = pyreadr.read_r("all_aml_anno.rds")[None]

i_all = all_aml_anno.index[all_aml_anno["ALL.AML"] == "AML"]   # note: same naming quirk as the R original

## 0 – Introduction and objectives

Last time we covered hypothesis testing and the t-test. This week: **non-parametric tests** and
**proportion testing**.

## 1 – Non-parametric test

What if the data is not normally distributed? Then a t-test isn't valid. Let's check the
distribution of gene FOSB:

In [ ]:
expression = all_aml.loc["FOSB"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(expression, bins=30)
stats.probplot(expression, dist="norm", plot=axes[1])
plt.show()

# Everything but normal!

Since the data isn't normal, we can't use the t-test — we need a **non-parametric test**: the
**Wilcoxon rank-sum test** (a.k.a. Mann-Whitney U test), `scipy.stats.mannwhitneyu()`. Like
Spearman vs. Pearson correlation, this test works on **ranks**, not raw values.

In [ ]:
exp_all = all_aml.loc["FOSB", i_all]
exp_aml = all_aml.loc["FOSB"].drop(index=i_all)

# R: wilcox.test(exp.all, exp.aml)
result_wilcox = stats.mannwhitneyu(exp_all, exp_aml)
print(result_wilcox)

In [ ]:
# Compare with the (invalid, here) t-test p-value:
result_ttest = stats.ttest_ind(exp_all, exp_aml)
print(result_ttest)

# The p-values are very different! We CANNOT trust the t-test here due to non-normality —
# the correct p-value is the one from the Wilcoxon test.
# Try another random gene!

## 2 – Proportion tests

The t-test and Wilcoxon test are **tests of the mean**. But sometimes we want to test the
relationship between two **categorical** variables — using a **contingency table** and Fisher's
exact test or the chi-square test.

In [ ]:
dat_brca = pd.read_csv(
    "https://www.dropbox.com/scl/fi/gy2jnj7o7p1398azkw6b1/gbsg_ba_ca.dat?rlkey=a0j003a1ehtt68usuyd32ta56&dl=1",
    sep="\t"
)

Is the choice of tamoxifen treatment (`hormon`) related to menopausal status (`meno`)? First, the contingency table:

In [ ]:
# R: table(dat.brca$meno, dat.brca$hormon)
ct = pd.crosstab(dat_brca["meno"], dat_brca["hormon"])
ct

The **odds-ratio (OR)**:

In [ ]:
OR = (ct.iloc[0, 0] / ct.iloc[0, 1]) / (ct.iloc[1, 0] / ct.iloc[1, 1])
OR

# How would the odds-ratio look if you transposed the table?

Now the one-sided Fisher Exact Test:

> H0: the odds-ratio is not significantly larger than one
> H1: the odds-ratio is significantly larger than one

In [ ]:
# R: fisher.test(tab, alternative='greater')
odds_ratio, p_value = stats.fisher_exact(ct, alternative="greater")
print("odds ratio:", odds_ratio, " p-value:", p_value)

# Check that this odds-ratio matches your manual computation above!
# Now run the two-sided Fisher test — first formulate the hypothesis!
# Check the p-values: what do you observe?

We can also use the **chi-square test**, comparing observed vs. expected counts under
independence:

> H0: observed and expected occurrences are not significantly different
> H1: observed and expected occurrences are significantly different

In [ ]:
# R: chisq.test(tab)
chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f"chi2={chi2}, p-value={p}, dof={dof}")

# Is this a one-sided or two-sided test?

Now: does age (over/under 40) affect tumor grade?

In [ ]:
# R: table(dat.brca$age>40, dat.brca$grade)
tab = pd.crosstab(dat_brca["age"] > 40, dat_brca["grade"])
tab

In [ ]:
chi2, p, dof, expected = stats.chi2_contingency(tab)
print(f"chi2={chi2}, p-value={p}, dof={dof}")

In [ ]:
# R: tot = apply(tab, 2, sum)  — column totals
tot = tab.sum(axis=0)
print(tot)

# R: age = apply(tab, 1, sum)  — row totals
age_totals = tab.sum(axis=1)
print(age_totals)

In [ ]:
tot_proportions = tot / tot.sum()

# R: tab.exp = sapply(tot.proportions, function(x) x * age)
tab_exp = pd.DataFrame(
    np.outer(age_totals, tot_proportions),
    index=tab.index, columns=tab.columns
)
tab_exp

# How would you compute the chi-square test statistic manually from `tab` and `tab_exp`?
# (Hint: this is exactly what `expected` from chi2_contingency() above already gives you.)

## 3 – Power of a test

$\alpha$ controls the **false-positive rate**. The other type of error, **false-negatives**, means
*not* seeing a difference that really exists — the **Type II error rate ($\beta$)**. Its complement
($1-\beta$) is the **power** (sensitivity) of the test.

Scenario: a biomarker test for a rare condition. Healthy: $\mu_H=50\,\mu g/L$, diseased:
$\mu_D=58\,\mu g/L$, both with $\sigma=12\,\mu g/L$. Each patient is measured $n=5$ times and the
average is used for diagnosis.

In [ ]:
mu_healthy = 50
mu_disease = 58
sigma = 12
n = 5
alpha = 0.05

patient_means = np.array([56.3, 62.1, 51.8, 49.2, 54.7])   # your first 5 patients today

For healthy individuals, the distribution of the mean of n=5 measurements is:

In [ ]:
x = np.arange(35, 65.1, 0.1)
y = stats.norm.pdf(x, loc=mu_healthy, scale=sigma / np.sqrt(n))

plt.plot(x, y, color="blue", linewidth=3)
plt.xlabel("Average biomarker level (μg/L)")
plt.ylabel("Density")
plt.title(f"Distribution of mean biomarker levels in healthy individuals (n={n})")
plt.show()

Using a one-sided test with alpha=0.05, when do we reject H0 ("patient is healthy")?

In [ ]:
# R: qnorm(p=alpha, mean=mu_healthy, sd=sigma/sqrt(n), lower.tail=FALSE)
rejection_threshold = stats.norm.ppf(1 - alpha, loc=mu_healthy, scale=sigma / np.sqrt(n))
rejection_threshold

In [ ]:
plt.plot(x, y, color="blue", linewidth=3)
plt.axvline(rejection_threshold, color="red", linestyle="--", linewidth=3)
for v in patient_means:
    plt.axvline(v, color="darkgreen", linestyle=":", linewidth=2)
plt.xlabel("Average biomarker level (μg/L)")
plt.ylabel("Density")
plt.title(f"Diagnostic decision rule (n={n})")
plt.show()

# Based on this threshold, patient 2 (62.1) would be diagnosed positive, the other 4 negative.

## The problem: false negatives in diseased patients

**But** all 5 of these patients actually **do** have the condition (confirmed by genetic testing)!
We missed 4 out of 5 — an 80% false-negative rate for this tiny sample. Let's simulate 1000 diseased
patients to estimate $\beta$ properly:

In [ ]:
rng = np.random.default_rng(0)

# Simulate 1000 diseased patients (n measurements each)
patient_means_sim = np.array([
    rng.normal(loc=mu_disease, scale=sigma, size=n).mean()
    for _ in range(1000)
])

rejection_threshold = stats.norm.ppf(1 - alpha, loc=mu_healthy, scale=sigma / np.sqrt(n))

# H0 (healthy) is rejected — disease correctly detected — if mean is ABOVE the threshold
beta = np.mean(patient_means_sim < rejection_threshold)

print(f"False-negative rate (β) = {beta*100:.1f}%")
print(f"Test power/sensitivity (1-β) = {(1-beta)*100:.1f}%")

A **very poor sensitivity** for a diagnostic test! Let's visualize the Type II error region:

In [ ]:
x = np.arange(35, 75.1, 0.1)
y_healthy = stats.norm.pdf(x, loc=mu_healthy, scale=sigma / np.sqrt(n))
y_disease = stats.norm.pdf(x, loc=mu_disease, scale=sigma / np.sqrt(n))

fig, ax = plt.subplots()
ax.plot(x, y_healthy, color="blue", linewidth=3, label="Healthy patients")
ax.plot(x, y_disease, color="purple", linewidth=3, label="Diseased patients")
ax.axvline(rejection_threshold, color="red", linestyle="--", linewidth=3, label="Diagnostic threshold")

x_fn = x[x < rejection_threshold]
y_fn = stats.norm.pdf(x_fn, loc=mu_disease, scale=sigma / np.sqrt(n))
ax.fill_between(x_fn, y_fn, color="purple", alpha=0.3, label="False-negative region")

ax.set_xlabel("Average biomarker level (μg/L)")
ax.set_ylabel("Density")
ax.set_title("Understanding false-negatives in diagnostic testing")
ax.legend()
plt.show()

# The shaded area: diseased patients whose average falls below the threshold — missed diagnoses.

> How does β change as n increases? Try n = 5, 10, 20, 50.
>
> Try to find a combination of effect size, alpha, and n that gives at least 80% power (β < 20%)
> while keeping alpha = 0.05.

---
## Exercises

### Exercise 1

1. Check the expression of a random gene in the ALL/AML dataset. Are the values normally distributed?
2. Run both a t-test and a Wilcoxon test on this gene. Is there a difference between the p-values?
   What did you expect?
3. Run both tests on **every** gene and store each p-value.
4. Scatter-plot -log10(p-value of t-test) vs. -log10(p-value of Wilcoxon test). Are there strong deviations?

In [ ]:
# Your code here:

### Exercise 2

What test would you use for the following questions?

- A lotion company wants to know whether their product is more likely to cause acne in men than in women
- The education department wants to know whether social science students have higher grades than science students
- A biologist wants to know whether a specific gene is more likely to be silenced in lactose-intolerant people

In [ ]:
# Your code here (as text/markdown reasoning is fine):

### Exercise 3

A pharmaceutical company's new vaccine works on 178 out of 200 patients. 75% of the batch had been
previously vaccinated with another vaccine, and it worked on 143 of those. Is there a significant
disproportion suggesting higher efficacy in previously vaccinated people, at alpha=0.05?
*(Hint: build a 2×2 contingency table and use `stats.fisher_exact` or `stats.chi2_contingency`.)*

In [ ]:
# Your code here:

## Going further: why non-normality breaks alpha as the false-positive rate

Suppose H0 ("the two distributions have equal expectation") is true: we draw two samples from the
**same** distribution and t-test them. What should the p-value distribution look like, over many
repetitions?

In [ ]:
rng = np.random.default_rng(123)
alpha = 0.05

p = np.array([
    stats.ttest_ind(rng.normal(size=10), rng.normal(size=10)).pvalue
    for _ in range(10000)
])

plt.hist(p, bins=20)
plt.axvline(alpha, color="red", linewidth=3)
plt.show()

In [ ]:
# Approximately alpha of these p-values should be below alpha (the red line):
print((p < alpha).mean())

In [ ]:
alphas = [0.01, 0.02, 0.03, 0.04, 0.05, 0.07, 0.1, 0.2]
fpr = [(p < a).mean() for a in alphas]

plt.scatter(alphas, fpr, color="red")
plt.plot([0, 0.2], [0, 0.2], color="lightgrey", linestyle="--")
plt.xlabel("alpha")
plt.ylabel("False-positive rate")
plt.show()

# With normal data, alpha really does equal the false-positive rate.

Now let's repeat this with **non-normal** data — samples from a t-distribution with df=1 (very heavy tails):

In [ ]:
p = np.array([
    stats.ttest_ind(rng.standard_t(df=1, size=10), rng.standard_t(df=1, size=10)).pvalue
    for _ in range(10000)
])

plt.hist(p, bins=20)
plt.axvline(alpha, color="red", linewidth=3)
plt.show()

# No longer a uniform distribution!

In [ ]:
print((p < alpha).mean())   # no longer close to alpha!

In [ ]:
alphas = [0.01, 0.02, 0.03, 0.04, 0.05, 0.07, 0.1, 0.2]
fpr = [(p < a).mean() for a in alphas]

plt.scatter(alphas, fpr, color="red")
plt.plot([0, 0.2], [0, 0.2], color="lightgrey", linestyle="--")
plt.xlabel("alpha")
plt.ylabel("False-positive rate")
plt.show()

# For non-normal data, the false-positive rate no longer equals alpha —
# we can no longer control the FPR with alpha!

## Summary: What have we learned?

| R | Python | Purpose |
|---|--------|---------|
| `wilcox.test(x, y)` | `scipy.stats.mannwhitneyu(x, y)` | Wilcoxon / Mann-Whitney non-parametric test |
| `table(x, y)` | `pd.crosstab(x, y)` | Contingency table |
| `fisher.test(tab, alternative=)` | `scipy.stats.fisher_exact(tab, alternative=)` | Fisher's exact test (2×2 tables) |
| `chisq.test(tab)` | `scipy.stats.chi2_contingency(tab)` | Chi-square test of independence |
| `apply(tab, 2, sum)` / `apply(tab, 1, sum)` | `tab.sum(axis=0)` / `tab.sum(axis=1)` | Row/column totals |
| `qnorm(p, mean, sd, lower.tail=FALSE)` | `stats.norm.ppf(1-p, loc, scale)` | Upper-tail quantile |
| `rt(n, df)` | `rng.standard_t(df, size=n)` | Random draws from a t-distribution |
| `rnorm(n, mean, sd)` | `rng.normal(loc, scale, size=n)` | Random normal draws |